# Day 4, Part 2a: The Groq Sandbox

**MIDAS Summer Academy**

This is a playground. The goal is to get comfortable calling a large language model through the Groq API and to build intuition for how the call's **parameters** and your **prompt** change what you get back. There is nothing to hand in. Run the cells, then change things and run them again.

In Part 2b you will point this same tool at a real research problem, so spend a few minutes here poking at it first.

## Setup

### Step 1: Install the Groq client

In [ ]:
!pip install -q groq

### Step 2: Create a client with your API key

You made a free key at [console.groq.com/keys](https://console.groq.com/keys). Paste it when prompted; it is not stored in the notebook.

(The Groq console shows `client = Groq()`, which reads the key from an environment variable. Here we pass it in directly so it works in a notebook.)

In [ ]:
from getpass import getpass
from groq import Groq

client = Groq(api_key=getpass("Paste your Groq API key: "))
print("Client ready.")

## Your first call

This is essentially the code the Groq console gives you, with a real question filled in. Run it and watch the answer stream in one piece at a time.

Then **edit it**: change the `content`, change `temperature`, change `max_completion_tokens`, and re-run.

In [ ]:
completion = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": "Explain why the sky is blue, in two sentences."}
    ],
    temperature=1,
    max_completion_tokens=8192,
    top_p=1,
    reasoning_effort="medium",
    stream=True,
    stop=None,
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")

### What each part of the call means

| Part | What it controls |
|---|---|
| `model` | which hosted model answers (here, an open 120B model) |
| `messages` | the conversation. Each has a `role` (`user`, `assistant`, or `system`) and `content` |
| `temperature` | randomness, 0 to 2. Low is focused and repeatable, high is varied |
| `max_completion_tokens` | a ceiling on how long the reply can be |
| `top_p` | an alternative randomness control; leave at 1 while changing temperature |
| `reasoning_effort` | how much this model "thinks" before answering: `low`, `medium`, `high` |
| `stream` | if `True`, tokens arrive as they are generated; if `False`, you get the whole reply at once |
| `stop` | optional text that, if produced, ends the reply early |

The streaming loop reads each `chunk` as it arrives. Some chunks carry no text, so `chunk.choices[0].delta.content or ""` prints an empty string for those.

## A helper so you can experiment quickly

Rather than copy the whole call every time, we wrap it in a small function. Every experiment below is a one-line call to `chat(...)` where you change one parameter and see what happens.

In [ ]:
MODEL = "openai/gpt-oss-120b"

def chat(prompt, system=None, temperature=1.0, max_completion_tokens=1024,
         reasoning_effort="medium", stream=True):
    # Same call as above, wrapped so you can change one setting at a time.
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    completion = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_completion_tokens=max_completion_tokens,
        top_p=1,
        reasoning_effort=reasoning_effort,
        stream=stream,
    )
    if not stream:
        return completion.choices[0].message.content
    text = ""
    for chunk in completion:
        piece = chunk.choices[0].delta.content or ""
        print(piece, end="")
        text += piece
    print()
    return text

_ = chat("Say hello in one short sentence.")

## Experiment 1: change the prompt

The single biggest lever is what you ask for. Try a vague request, then a specific one, and notice how much more useful the specific answer is.

In [ ]:
_ = chat("Tell me about caffeine.")
print("\n" + "-" * 40 + "\n")
_ = chat("In 3 bullet points, summarize how caffeine affects reaction time. Keep it under 40 words.")

## Experiment 2: temperature

Temperature controls randomness. At `0` the same prompt gives essentially the same answer every time. At a high value it varies. Run this cell a couple of times and compare.

In [ ]:
print("temperature = 0 (repeatable):")
_ = chat("Invent a two-word name for a coffee shop.", temperature=0)
_ = chat("Invent a two-word name for a coffee shop.", temperature=0)

print("\ntemperature = 1.5 (varied):")
_ = chat("Invent a two-word name for a coffee shop.", temperature=1.5)
_ = chat("Invent a two-word name for a coffee shop.", temperature=1.5)

## Experiment 3: max_completion_tokens

This caps the length of the reply. Set it low and watch the answer get cut off mid-thought.

In [ ]:
_ = chat("Write a detailed paragraph about the ocean.", max_completion_tokens=40)

## Experiment 4: reasoning_effort

This model can spend more or less effort "thinking" before it answers. On an easy-to-trip-up question, compare `low` and `high`. (Higher effort is usually slower.)

In [ ]:
puzzle = ("A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
          "How much does the ball cost? Answer with just the amount.")

print("reasoning_effort = low:")
_ = chat(puzzle, reasoning_effort="low")

print("\nreasoning_effort = high:")
_ = chat(puzzle, reasoning_effort="high")

## Experiment 5: a system message

A `system` message sets the model's role or style for the whole conversation. Same question, different instructions.

In [ ]:
_ = chat("Explain photosynthesis.", system="You are a scientist. Be precise and use one sentence.")
print("\n" + "-" * 40 + "\n")
_ = chat("Explain photosynthesis.", system="You reply only in rhyming couplets.")

## Experiment 6: turn off streaming, and see token usage

With `stream=False` you get the whole reply at once as a normal object. That object also tells you how many **tokens** the call used, which is what you are billed on. Part 2b uses this.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Name three unusual uses for a paperclip."}],
    temperature=1,
    max_completion_tokens=256,
    stream=False,
)

print(response.choices[0].message.content)
print("\ntokens:", response.usage.prompt_tokens, "in +", response.usage.completion_tokens, "out")

## Your turn

Try to make the model do each of these by changing prompts and parameters:

1. Answer a question in exactly one word.
2. Produce the same answer three times in a row (which temperature makes this reliable?).
3. Respond as if it were a cautious peer reviewer.
4. Return its answer as a JSON object with keys `summary` and `keywords`.
5. Get a noticeably better answer to a tricky question by raising `reasoning_effort`.

When you are comfortable driving the API, move on to **Part 2b**, where we use it to code a corpus of research abstracts and then ask the harder question: can we trust what it produced?